chapter six: photographing the invisible

In [1]:
import numpy as np
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
import time

M = 1.0
R_S = 2*M
# throwback to chapter 1
b_crit = 3*np.sqrt(3)*M          

plt.rcParams['figure.dpi'] = 110

def camera_rays(width, height, r_cam=45.0, incl_deg=17.0, fov_deg=24.0):
    th = np.radians(incl_deg)
    cam = np.array([np.sin(th), 0.0, np.cos(th)]) * r_cam
    fwd = -cam / np.linalg.norm(cam)
    # looking straight down the axis
    if abs(incl_deg) < 0.5:                       
        right = np.array([0., 1., 0.])
    else:
        right = np.cross(fwd, [0., 0., 1.]); right /= np.linalg.norm(right)
    up = np.cross(right, fwd)
    tanf = np.tan(np.radians(fov_deg)/2)
    xs = np.linspace(-1, 1, width)*tanf
    ys = np.linspace(-1, 1, height)*tanf*(height/width)
    X, Y = np.meshgrid(xs, ys)
    d = fwd[None,None,:] + X[...,None]*right + Y[...,None]*up
    d /= np.linalg.norm(d, axis=-1, keepdims=True)
    return cam, d.reshape(-1,3)

In [ ]:
def trace(cam, dirs, r_in, r_out, dphi=0.005, n_steps=4000, r_escape_mult=2.5):
    """the binet tracer, now with a memory.
    returns: hit_r, hit_xy (landing spot), Lz_back (for doppler),
             esc_dir (escapees' final direction), order (plane crossings before landing)."""
    N = dirs.shape[0]
    pos = np.broadcast_to(cam, (N,3)).astype(np.float64)
    r0 = np.linalg.norm(cam)
    e1 = pos / r0
    vr = (dirs*e1).sum(1)
    tang = dirs - vr[:,None]*e1
    vt = np.maximum(np.linalg.norm(tang, axis=1), 1e-12)
    e2 = tang / vt[:,None]
    Lz_back = np.cross(pos, dirs)[:,2]
    u = np.full(N, 1.0/r0); up_ = -vr/(r0*vt); phi = np.zeros(N)
    z_prev, u_prev = pos[:,2].copy(), u.copy()
    hit_r  = np.full(N, np.nan)
    hit_xy = np.full((N,2), np.nan)
    esc_dir = np.full((N,3), np.nan)
    order  = np.zeros(N, dtype=int)
    active = np.ones(N, bool)
    # still the whole renderer
    rhs = lambda u: 3*M*u*u - u                     

    for _ in range(n_steps):
        if not active.any():
            break
        a = active.copy()
        ua, upa = u[a], up_[a]
        k1u,k1p = upa, rhs(ua)
        k2u,k2p = upa+0.5*dphi*k1p, rhs(ua+0.5*dphi*k1u)
        k3u,k3p = upa+0.5*dphi*k2p, rhs(ua+0.5*dphi*k2u)
        k4u,k4p = upa+dphi*k3p,     rhs(ua+dphi*k3u)
        u[a]   = ua  + dphi/6*(k1u+2*k2u+2*k3u+k4u)
        up_[a] = upa + dphi/6*(k1p+2*k2p+2*k3p+k4p)
        phi[a] += dphi
        ca, sa = np.cos(phi[a]), np.sin(phi[a])
        za = (ca*e1[a,2] + sa*e2[a,2]) / u[a]
        idx = np.where(a)[0]

        crossed = np.sign(za) != np.sign(z_prev[a])
        if crossed.any():
            frac = np.abs(z_prev[a])/(np.abs(z_prev[a])+np.abs(za)+1e-30)
            u_c  = u_prev[a] + frac*(u[a]-u_prev[a])
            ph_c = phi[a] - dphi*(1-frac)
            r_c  = 1.0/u_c
            hit = crossed & (r_c >= r_in) & (r_c <= r_out)
            # passed through the transparent plane
            order[idx[crossed & ~hit]] += 1          
            hidx = idx[hit]
            hit_r[hidx] = r_c[hit]
            cc, ss = np.cos(ph_c[hit]), np.sin(ph_c[hit])
            hit_xy[hidx,0] = (cc*e1[hidx,0] + ss*e2[hidx,0]) * r_c[hit]
            hit_xy[hidx,1] = (cc*e1[hidx,1] + ss*e2[hidx,1]) * r_c[hit]
            active[hidx] = False

        captured = u[a] > 1.0/(R_S*1.01)
        escaped  = (u[a] < 1.0/(r_escape_mult*r0)) & (up_[a] < 0)
        eidx = idx[escaped]
        # record the getaway direction
        if eidx.size:                                
            ce, se = np.cos(phi[eidx]), np.sin(phi[eidx])
            v = (u[eidx,None]*(-se[:,None]*e1[eidx] + ce[:,None]*e2[eidx])
                 - up_[eidx,None]*(ce[:,None]*e1[eidx] + se[:,None]*e2[eidx]))
            esc_dir[eidx] = v / np.linalg.norm(v, axis=1, keepdims=True)
        active[idx[captured | escaped]] = False
        z_prev[a], u_prev[a] = za, u[a]
    return hit_r, hit_xy, Lz_back, esc_dir, order

def blackbody_rgb(T):
    T = np.clip(T, 1000, 40000)/100
    r = np.where(T<=66, 255, 329.7*np.clip(T-60,1e-3,None)**-0.1332)
    g = np.where(T<=66, 99.47*np.log(T)-161.1, 288.1*np.clip(T-60,1e-3,None)**-0.0755)
    b = np.where(T>=66, 255, np.where(T<=19, 0, 138.5*np.log(np.clip(T-10,1e-3,None))-305.0))
    return np.clip(np.stack([r,g,b],-1)/255, 0, 1)

def shade(hit_r, Lz_back, shape, r_in, T_inner=9000.0):
    img = np.zeros((hit_r.size, 3))
    hit = np.isfinite(hit_r)
    r, lam = hit_r[hit], -Lz_back[hit]
    Omega = np.sqrt(M/r**3)
    g = np.sqrt(np.clip(1-3*M/r, 0, None)) / (1 + Omega*lam)
    flux = np.clip(r**-3*(1-np.sqrt(np.minimum(r_in/r,1.0))), 0, None)
    I = g**4*flux; I /= np.percentile(I[I>0], 99.5)
    rgb = blackbody_rgb(T_inner*(r/r_in)**-0.75*g)
    img[hit] = rgb*np.clip(1-np.exp(-3.0*I),0,1)[:,None]**0.85
    g_full = np.full(hit_r.size, np.nan); g_full[hit] = g
    return img.reshape(*shape,3), g_full